# Compute-cost probe — BIO vs QA-pointer, span-only and multi-task pairs

`idiombert_contributions.md`'s C1 claims BIO is "simpler tagging schema + less compute" than QA-pointer — that claim was **never measured**, only asserted (flagged 2026-06-25). The multi-task pair (E4 vs System E) is the one that matters — same training paradigm, only span-head formulation differs, the clean single-confound comparison. Span-only (BIO run16 vs QA System A) was attempted and abandoned (out of scope, run did not persist, not debugged).

- **BIO span-only** — `run_16_wordpiece_word_tagger.py` (word-level B/I/O tagger, no classifier head)
- **BIO multi-task (E4)** — `run_17_bio_cls_joint.py` (BIO span head + CLS idiomaticity head, jointly trained)
- **QA multi-task (System E)** — `training/Train_Join.py` (`JointIdiomModel`, QA-pointer span head + CLS head, jointly trained — the exact single-confound counterpart to E4: same paradigm, only span-head formulation differs)

Measures, per system: (1) wall-clock seconds/epoch and total training time, (2) peak GPU memory (nvidia-smi-polled every 2s during the subprocess training run, MiB — coarse, not torch.cuda's exact allocator peak, but sufficient for a single-seed point comparison), same seed (42), same canonical hyperparams as each system's registered config, single run each (timing/memory are not statistical metrics, no 3-seeding needed).

**Does NOT touch canonical artifacts.** Writes to `_timing_*_s42` output dirs, distinct from canonical `word_tagger_mbert_s42` / `bio_cls_joint_mbert_s42` / `en_es_hi_te/joint_mbert` — those stay untouched. This run's numbers are for the compute-cost question only, not a re-registration of accuracy metrics (ignore exact/overlap/F1/cls_f1 printed here, they're incidental).

**Persistence:** outputs symlinked to Drive, hard-fails before training if not Drive-backed (same gate as every other runner in this repo). Already-completed runs (BIO/E4/System E timing) need to be **re-run** to pick up the new peak-memory field — old `timing.json` files don't have it.

In [ ]:
# 1. Config
REPO_URL = 'https://github.com/JustLetMeBeHello/Idiomator_Research.git'
BRANCH   = 'main'
REPO     = '/content/Idiomator_Research'   # absolute — never use a relative %cd
SEED     = '42'
DRIVE_OUT = '/content/drive/MyDrive/IdiomatorRigor'
print('repo:', REPO, '| seed:', SEED)

In [ ]:
# 2. Clone / refresh repo, pin absolute cwd, kill any nested duplicate clone
import os, subprocess, sys
from pathlib import Path
nested = os.path.join(REPO, 'Idiomator_Research')
if os.path.isdir(nested):
    subprocess.run(['rm', '-rf', nested], check=True)
if os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '--quiet', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '--quiet', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--quiet', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--quiet', '--branch', BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
print('cwd:', os.getcwd())

In [ ]:
# 3. Install deps + confirm GPU
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'Requirements.txt'], check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU — switch runtime to T4/A100'

In [ ]:
# 4. Mount Drive, make timing-only output dirs, symlink them (Drive-backed,
#    separate namespace from canonical model dirs)
from google.colab import drive
drive.mount('/content/drive')
Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)

for name in ('_timing_bio_s42', '_timing_e4bio_s42', '_timing_qaspan_s42', '_timing_e_s42'):
    drive_path = Path(DRIVE_OUT) / name
    drive_path.mkdir(parents=True, exist_ok=True)
    local_path = Path('models') / name
    if local_path.is_symlink() or local_path.exists():
        if local_path.is_symlink():
            local_path.unlink()
        else:
            raise RuntimeError(f'{local_path} exists and is not a symlink — refusing to overwrite a real dir')
    os.symlink(str(drive_path), str(local_path))
    assert os.path.islink(local_path) and 'drive' in os.readlink(local_path).lower(), \
        f'{local_path} not Drive-backed — refusing to let training write to ephemeral /content'
    print(f'{local_path} -> {os.readlink(local_path)}  (Drive-backed: OK)')

In [ ]:
# 5. Param-count probe — load both model classes (no data needed), count
#    trainable params, and isolate the head-only delta vs the shared mBERT
#    backbone. This answers "is BIO actually fewer params" independent of
#    training time.
import importlib.util, torch

def load_module(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

m16 = load_module('experiments/rigor/run_16_wordpiece_word_tagger.py', 'run16')
m17 = load_module('experiments/rigor/run_17_bio_cls_joint.py', 'run17')

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Both scripts build their model class on top of the same backbone string —
# inspect each module for its model class and instantiate with defaults.
bio_model_cls = [v for k, v in vars(m16).items() if isinstance(v, type) and 'Module' in [c.__name__ for c in v.__mro__]][0]
e4_model_cls  = [v for k, v in vars(m17).items() if isinstance(v, type) and 'Module' in [c.__name__ for c in v.__mro__]][0]

print('BIO single-task model class:', bio_model_cls.__name__)
print('E4 multi-task model class:  ', e4_model_cls.__name__)
print()
print('NOTE: if instantiation below fails (constructor args differ), this cell')
print('is best-effort — fall back to reading param counts from training logs')
print('in cells 6/7 instead (most HF-style trainers print this on startup).')

In [ ]:
# 6. TIMED RUN — BIO single-task (run_16), seed 42, canonical hyperparams
#    (epochs=6, batch=32, lr=3.27e-5 — matches the registered word_tagger_mbert_s42 config).
#    Subprocess training, so GPU peak memory is read from `nvidia-smi` polled by
#    a background thread (torch.cuda stats live in the subprocess, not this kernel).
import time, json, threading

def poll_peak_mem_mb(stop_event, peak_holder):
    peak = 0
    while not stop_event.is_set():
        try:
            out = subprocess.run(['nvidia-smi', '--query-gpu=memory.used',
                                   '--format=csv,noheader,nounits'],
                                  capture_output=True, text=True, timeout=5)
            used = int(out.stdout.strip().splitlines()[0])
            peak = max(peak, used)
        except Exception:
            pass
        stop_event.wait(2)
    peak_holder['peak_mb'] = peak

stop_evt = threading.Event()
peak_holder = {'peak_mb': 0}
poller = threading.Thread(target=poll_peak_mem_mb, args=(stop_evt, peak_holder))
poller.start()

t0 = time.time()
result = subprocess.run([
    sys.executable, 'experiments/rigor/run_16_wordpiece_word_tagger.py',
    '--output_dir', 'models/_timing_bio_s42',
    '--epochs', '6', '--batch_size', '32', '--lr', '3.27e-5',
    '--seed', SEED,
], capture_output=True, text=True)
elapsed_bio = time.time() - t0
stop_evt.set()
poller.join()

print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-3000:])
print(f'\nBIO single-task wall-clock: {elapsed_bio:.1f}s ({elapsed_bio/60:.2f} min) for 6 epochs')
print(f'BIO single-task peak GPU memory: {peak_holder["peak_mb"]} MiB')

Path('models/_timing_bio_s42/timing.json').write_text(json.dumps({
    'system': 'bio_single_task_run16', 'seed': int(SEED), 'epochs': 6,
    'wall_clock_seconds': elapsed_bio, 'peak_gpu_mem_mb': peak_holder['peak_mb'],
    'returncode': result.returncode,
}, indent=2))
print('timing written to Drive: models/_timing_bio_s42/timing.json')

In [ ]:
# 7. TIMED RUN — E4 BIO+CLS multi-task (run_17), seed 42, canonical hyperparams
#    (epochs=7, batch=32, lr=2e-5 — matches the registered bio_cls_joint_mbert_s42 config).
#    Reuses poll_peak_mem_mb from cell 6 — run cell 6 first in this session.
stop_evt = threading.Event()
peak_holder = {'peak_mb': 0}
poller = threading.Thread(target=poll_peak_mem_mb, args=(stop_evt, peak_holder))
poller.start()

t0 = time.time()
result = subprocess.run([
    sys.executable, 'experiments/rigor/run_17_bio_cls_joint.py',
    '--output_dir', 'models/_timing_e4bio_s42',
    '--epochs', '7', '--batch_size', '32', '--lr', '2e-5',
    '--seed', SEED,
], capture_output=True, text=True)
elapsed_e4 = time.time() - t0
stop_evt.set()
poller.join()

print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-3000:])
print(f'\nE4 BIO multi-task wall-clock: {elapsed_e4:.1f}s ({elapsed_e4/60:.2f} min) for 7 epochs')
print(f'E4 BIO multi-task peak GPU memory: {peak_holder["peak_mb"]} MiB')

Path('models/_timing_e4bio_s42/timing.json').write_text(json.dumps({
    'system': 'e4_bio_cls_joint_run17', 'seed': int(SEED), 'epochs': 7,
    'wall_clock_seconds': elapsed_e4, 'peak_gpu_mem_mb': peak_holder['peak_mb'],
    'returncode': result.returncode,
}, indent=2))
print('timing written to Drive: models/_timing_e4bio_s42/timing.json')

In [ ]:
# 10. Persistence readback + matched-pair comparison — reads timing.json back
#     FROM DRIVE (not /content), normalizes to seconds/epoch, compares each BIO
#     system against its matched-structure QA counterpart. Also reports peak
#     GPU memory (MiB, polled via nvidia-smi every 2s during training — coarse,
#     not torch.cuda's exact allocator peak, but fine for a single-seed point
#     comparison).
import json
rows = {}
for name, label in [('_timing_bio_s42', 'BIO span-only (run16)'),
                     ('_timing_qaspan_s42', 'QA span-only, System A (Stage_2)'),
                     ('_timing_e4bio_s42', 'BIO multi-task, E4 (run17)'),
                     ('_timing_e_s42', 'QA multi-task, System E (Train_Join)')]:
    p = Path(DRIVE_OUT) / name / 'timing.json'
    if not p.exists():
        print(f'{label}: NOT FOUND at {p} — run did not persist')
        continue
    d = json.loads(p.read_text())
    per_epoch = d['wall_clock_seconds'] / d['epochs']
    peak_mb = d.get('peak_gpu_mem_mb', 'n/a')
    rows[name] = (label, d['wall_clock_seconds'], d['epochs'], per_epoch, peak_mb)
    print(f"{label}: total={d['wall_clock_seconds']:.1f}s  epochs={d['epochs']}  "
          f"sec/epoch={per_epoch:.1f}  peak_gpu_mem={peak_mb} MiB")

print()
if '_timing_bio_s42' in rows and '_timing_qaspan_s42' in rows:
    bio, qa = rows['_timing_bio_s42'], rows['_timing_qaspan_s42']
    diff_pct = (qa[3] - bio[3]) / bio[3] * 100
    print(f"Span-only pair: QA (System A) is {diff_pct:+.1f}% sec/epoch vs BIO (run16).")
if '_timing_e4bio_s42' in rows and '_timing_e_s42' in rows:
    e4, e = rows['_timing_e4bio_s42'], rows['_timing_e_s42']
    diff_pct = (e[3] - e4[3]) / e4[3] * 100
    print(f"Multi-task pair: QA (System E) is {diff_pct:+.1f}% sec/epoch vs BIO (E4).")
    if isinstance(e4[4], int) and isinstance(e[4], int) and e4[4] > 0:
        mem_diff_pct = (e[4] - e4[4]) / e4[4] * 100
        print(f"Multi-task pair: QA (System E) peak GPU mem is {mem_diff_pct:+.1f}% vs BIO (E4) "
              f"({e[4]} MiB vs {e4[4]} MiB).")

print('\nReminder: ONE seed each, no variance estimate — report as point measurements,')
print('not statistically tested claims, unless re-run at 3 seeds. GPU memory is')
print('nvidia-smi-polled (2s granularity), not an exact allocator peak.')

In [ ]:
# 9. TIMED RUN — QA multi-task, System E (training/Train_Join.py), seed 42,
#    canonical hyperparams (epochs=7, batch=32, lr=2e-5, cls=0.3/span=1.9 —
#    matches the registered en_es_hi_te/joint_mbert config). This is the exact
#    single-confound counterpart to E4 (cell 7): same paradigm, only the span
#    head differs (QA pointer vs BIO tag). Reuses poll_peak_mem_mb from cell 6.
stop_evt = threading.Event()
peak_holder = {'peak_mb': 0}
poller = threading.Thread(target=poll_peak_mem_mb, args=(stop_evt, peak_holder))
poller.start()

t0 = time.time()
result = subprocess.run([
    sys.executable, 'training/Train_Join.py',
    '--output_dir', 'models/_timing_e_s42',
    '--epochs', '7', '--batch_size', '32', '--lr', '2e-5',
    '--cls_loss_weight', '0.3', '--span_loss_weight', '1.9',
    '--seed', SEED,
], capture_output=True, text=True)
elapsed_e = time.time() - t0
stop_evt.set()
poller.join()

print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-3000:])
print(f'\nQA multi-task (System E) wall-clock: {elapsed_e:.1f}s ({elapsed_e/60:.2f} min) for 7 epochs')
print(f'QA multi-task (System E) peak GPU memory: {peak_holder["peak_mb"]} MiB')

Path('models/_timing_e_s42/timing.json').write_text(json.dumps({
    'system': 'qa_cls_joint_systemE_TrainJoin', 'seed': int(SEED), 'epochs': 7,
    'wall_clock_seconds': elapsed_e, 'peak_gpu_mem_mb': peak_holder['peak_mb'],
    'returncode': result.returncode,
}, indent=2))
print('timing written to Drive: models/_timing_e_s42/timing.json')